## 전처리 다운로드 & Import

#### 크롤링

In [ ]:
! python -m pip install requests pandas beautifulsoup4
! python -m pip install selenium webdriver-manager
! python -m pip install pillow requests selenium webdriver-manager

#### pdf 변환

In [ ]:
! pip install opencv-python
! pip install matplotlib
! pip install pytesseract
! pip install pdfplumber
! pip install pytesseract pdf2image pillow

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from PIL import Image
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException
from webdriver_manager.chrome import ChromeDriverManager

import cv2
from matplotlib import pyplot as plt
import shutil
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

## 국방위 회의록 다운로드

In [ ]:
BASE_URL = "https://defense.na.go.kr:444/cmmit/cmitMtgRcord/mtgRcord/mtgRcordList.do"
PARAMS = {
    'menuNo': '2000071',
    'pageIndex': '1',
    'pageUnit': '50'  # 한 페이지에 50개를 불러오도록 설정
}
SAVE_DIR =  os.path.abspath("국방위_회의록")

# 저장할 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def download_minutes():
    print("데이터를 불러오는 중입니다...")
    
    # 웹페이지 요청 (SSL 인증서 검증 건너뛰기 -> .go.kr 사이트 특성 대응)
    response = requests.get(BASE_URL, params=PARAMS, verify=False)
    if response.status_code != 200:
        print("페이지 접속에 실패했습니다.")
        return

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 테이블 내의 모든 행(tr) 찾기
    # 이미지의 id="mtgRcordList-dataset-data-table" 내부의 tr을 타겟팅
    rows = soup.select("#mtgRcordList-dataset-data-table tr")
    
    print(f"총 {len(rows)}개의 항목을 발견했습니다.")

    for i, row in enumerate(rows):
        # 다운로드 링크가 있는 a 태그 찾기
        # 이미지 구조상 마지막 td 안의 a 태그에 링크가 있음
        link_tag = row.select_one("td a[href*='download/pdf']")
        
        if link_tag:
            download_url = link_tag['href']
            # 상대 경로일 경우 절대 경로로 변환
            if not download_url.startswith("http"):
                download_url = "https://defense.na.go.kr:444" + download_url
            
            # 파일명 설정 (행의 텍스트나 alt 속성을 활용 가능)
            # alt 속성에 있는 회의명을 파일명으로 사용
            img_tag = link_tag.find("img")
            file_name = img_tag['alt'].replace(" pdf 다운로드", "").strip() if img_tag else f"meeting_{i+1}"
            file_name = "".join(c for c in file_name if c.isalnum() or c in (' ', '_', '-')).rstrip() # 특수문자 제거
            full_path = os.path.join(SAVE_DIR, f"{file_name}.pdf")

            # 파일 다운로드
            try:
                print(f"[{i+1}/{len(rows)}] 다운로드 중: {file_name}")
                file_res = requests.get(download_url, verify=False)
                with open(full_path, 'wb') as f:
                    f.write(file_res.content)
            except Exception as e:
                print(f"실패: {file_name} (오류: {e})")

    print("\n모든 작업이 완료되었습니다.")

if __name__ == "__main__":
    # SSL 경고 메시지 무시 설정
    requests.packages.urllib3.disable_warnings()
    download_minutes()

## 국방위 보도자료 다운로드

In [ ]:
TARGET_URL = "https://defense.na.go.kr:444/cmmit/bbs/B0000051/list.do?pageIndex=1&menuNo=2000037&searchWrd=&searchCnd=1&sdate=&edate=&searchWrdMb=&sdateMb=&edateMb=&pageUnit=50"
SAVE_DIR = os.path.abspath("국방위_보도자료")

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 크롬 옵션 설정
chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": False # 안전하지 않은 파일 경고 무시(False)
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument('--ignore-certificate-errors') # SSL 인증서 오류 무시
chrome_options.add_argument('--ignore-ssl-errors')

# 드라이버 실행
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def download_files():
    try:
        driver.get(TARGET_URL)
        
        # 페이지 로딩을 확실히 기다리기 -> tbody가 나타날 때까지
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody")))
        
        rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
        print(f"총 {len(rows)}개의 항목을 발견했습니다.")

        if len(rows) == 0:
            print("데이터를 찾지 못했습니다. 브라우저 창에 페이지가 정상적으로 뜨는지 확인해주세요.")
            return

        for i in range(len(rows)):
            try:
                # 매 반복마다 요소를 새로 갱신 (StaleElementReferenceException 방지)
                current_rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
                row = current_rows[i]
                
                # 다운로드 버튼 찾기
                download_btn = row.find_element(By.CSS_SELECTOR, "a.btn_board_download")
                file_info = download_btn.get_attribute("title")
                print(f"[{i+1}/{len(rows)}] {file_info} 처리 중...")

                # 1) 자바스크립트 직접 실행하여 레이어 열기
                driver.execute_script("arguments[0].click();", download_btn)
                time.sleep(1)

                # 2) 레이어 내부의 실제 다운로드 링크 클릭
                real_links = row.find_elements(By.CSS_SELECTOR, ".board_download_layer a")
                
                for link in real_links:
                    driver.execute_script("arguments[0].click();", link)
                    time.sleep(1)
                
            except Exception as e:
                continue

        print(f"\n✅ 작업 완료! 파일 저장 경로: {SAVE_DIR}")
        time.sleep(5)

    finally:
        driver.quit()

if __name__ == "__main__":
    download_files()

## 국방위 국정감사 다운로드

In [ ]:
TARGET_URL = "https://defense.na.go.kr:444/cmmit/bbs/BCMT2003/list.do?pageIndex=1&menuNo=2000031&pageUnit=30"
SAVE_DIR = os.path.abspath("국방위_국정감사")

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": False
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument('--ignore-certificate-errors')
chrome_options.add_argument('--ignore-ssl-errors')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def start_download():
    try:
        driver.get(TARGET_URL)
        wait = WebDriverWait(driver, 15)
        
        # 로딩 대기
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.board01 table tbody tr")))
        
        # 실제 데이터가 있는 행의 인덱스를 미리 확보
        all_rows = driver.find_elements(By.CSS_SELECTOR, "div.board01 table tbody tr")
        data_indices = []
        for idx, r in enumerate(all_rows):
            # td가 1개인 행(공지/데이터없음)은 제외하고 실제 게시글만 인덱스 저장
            if len(r.find_elements(By.TAG_NAME, "td")) > 1:
                data_indices.append(idx)
        
        print(f"실제 게시글 개수: {len(data_indices)}개")

        for i, row_idx in enumerate(data_indices):
            try:
                # 매번 요소를 새로 찾아서 'row_idx'로 접근해야 밀리지 않음
                current_rows = driver.find_elements(By.CSS_SELECTOR, "div.board01 table tbody tr")
                row = current_rows[row_idx]
                
                try:
                    cells = row.find_elements(By.TAG_NAME, "td")
                    title_text = cells[1].text if len(cells) > 1 else "Unknown Title"
                except:
                    title_text = "제목 추출 실패"
                
                print(f"[{i+1}/{len(data_indices)}] 처리 중: {title_text}")

                download_btns = row.find_elements(By.CSS_SELECTOR, "a.btn_board_download")
                if not download_btns:
                    print(f"   -> 다운로드 버튼 없음")
                    continue

                driver.execute_script("arguments[0].click();", download_btns[0])
                time.sleep(1.2)

                # 파일 링크 추출 및 중복 제거
                raw_links = row.find_elements(By.CSS_SELECTOR, ".board_download_layer ul li a[onclick*='Download']")
                unique_links = {}
                for link in raw_links:
                    onclick_val = link.get_attribute("onclick")
                    if onclick_val not in unique_links:
                        unique_links[onclick_val] = (link, link.get_attribute("title"))

                if not unique_links:
                    print(f"   -> [참고] 레이어 내 파일 링크가 비어있음")

                for onclick_func, (link_obj, f_name) in unique_links.items():
                    try:
                        print(f"   -> 다운로드 실행: {f_name}")
                        driver.execute_script("arguments[0].click();", link_obj)
                        time.sleep(2) 

                        # 알림창(파일 없음 등)이 뜨면 즉시 닫기
                        try:
                            alert = driver.switch_to.alert
                            print(f"      ⚠️ 서버 알림: {alert.text}")
                            alert.accept()
                        except NoAlertPresentException:
                            pass

                    except UnexpectedAlertPresentException:
                        try:
                            alert = driver.switch_to.alert
                            alert.accept()
                        except: pass
                        continue

            except Exception as e:
                print(f"[{i+1}번 행] 에러 발생 (건너뜀 방지 처리): {e}")
                continue

        print(f"\n✅ 모든 게시글 확인 완료! 폴더: {SAVE_DIR}")

    finally:
        driver.quit()

if __name__ == "__main__":
    start_download()

## 국방부 정책자료 다운로드

In [ ]:
TARGET_URL = "https://www.mnd.go.kr/cop/pblictn/selectPublicationsUser.do?siteId=mnd&componentId=14&categoryId=18&pageIndex=1&id=mnd_020704000000"
SAVE_DIR = os.path.abspath("국방부_정책자료")

os.makedirs(SAVE_DIR, exist_ok=True)

chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "plugins.always_open_pdf_externally": True
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--ignore-ssl-errors")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

wait = WebDriverWait(driver, 15)

def start_download(max_page=3):  # 페이지 수 조절
    driver.get(TARGET_URL)

    for page in range(1, max_page + 1):
        print(f"\n📄 페이지 {page} 처리 중...")

        if page > 1:
            driver.execute_script(f"publicationSearch('{page}')")
            time.sleep(0.5)

        # 게시글 리스트 로딩
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "ul.list_post > li")
        ))

        posts = driver.find_elements(By.CSS_SELECTOR, "ul.list_post > li")
        print(f"   ▶ 게시글 수: {len(posts)}")

        for i in range(len(posts)):
            try:
                # DOM 재로딩
                posts = driver.find_elements(By.CSS_SELECTOR, "ul.list_post > li")
                post = posts[i]

                link = post.find_element(By.CSS_SELECTOR, "a")
                title = link.text.strip() or f"Unknown_{page}_{i}"

                print(f"   [{i+1}] {title}")

                # 상세 페이지로 이동
                driver.execute_script("arguments[0].click();", link)
                time.sleep(0.5)

                # 다운로드 링크(PDF 파일) 탐색
                download_links = driver.find_elements(
                    By.CSS_SELECTOR,
                    "a[href*='.pdf'], a[href*='download']"
                )

                if not download_links:
                    print("      ❌ PDF 없음")
                    driver.back()
                    time.sleep(0.2)
                    continue

                for dl in download_links:
                    try:
                        print("      ⬇ PDF 다운로드")
                        driver.execute_script("arguments[0].click();", dl)
                        time.sleep(1)

                        # 알림창 닫기 처리
                        try:
                            alert = driver.switch_to.alert
                            alert.accept()
                        except NoAlertPresentException:
                            pass

                    except Exception as e:
                        print(f"      ❌ 다운로드 오류: {e}")

                driver.back()
                time.sleep(0.5)

            except Exception as e:
                print(f"   ❌ 게시글 처리 오류: {e}")
                driver.back()
                time.sleep(0.5)

    print(f"\n✅ 다운로드 완료: {SAVE_DIR}")
    time.sleep(1)
    driver.quit()


if __name__ == "__main__":
    start_download(max_page=3)

#### -> 파일로 한 번에 다운로드 불가능할 때
 - 이미지로 다운로드 하여 pdf 한 묶음으로 만들 것

In [ ]:
IMAGE_DIR = "ebook_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

TOTAL_PAGES = 64
PDF_PATH = "국방개혁_2.0.pdf"

URL = "https://www.mnd.go.kr/mbshome/mbs/reform/reform_ebook/reform2.0/index.html"

driver = webdriver.Chrome()
driver.get(URL)

WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)
time.sleep(2)

body = driver.find_element(By.TAG_NAME, "body")

print("📸 페이지 캡처 시작")

image_paths = []

for page in range(1, TOTAL_PAGES + 1):
    filename = f"{page:03}.png"
    path = os.path.join(IMAGE_DIR, filename)

    # 화면 캡처 (현재 페이지)
    driver.save_screenshot(path)
    image_paths.append(path)

    print(f"✅ {page}페이지 캡처 완료")

    if page != TOTAL_PAGES:
        body.send_keys(Keys.ARROW_RIGHT)
        # 페이지 렌더링 대기
        time.sleep(0.6) 

driver.quit()

print("📄 PDF 생성 중...")

# PNG -> PDF
images = [Image.open(p).convert("RGB") for p in image_paths]
images[0].save(
    PDF_PATH,
    save_all=True,
    append_images=images[1:]
)

print(f"✨ 완료! PDF 생성됨: {PDF_PATH}")

## 국방기술진흥연구소 자료 다운로드

In [ ]:
TARGET_URL = "https://www.krit.re.kr/krit/bbs/gbgs_list.do?gotoMenuNo=03090100"
SAVE_DIR = os.path.abspath("국기연_발간물")

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": False,
    "plugins.always_open_pdf_externally": True  # PDF 바로 다운로드
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument('--ignore-certificate-errors')
chrome_options.add_argument('--ignore-ssl-errors')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def start_download():
    try:
        driver.get(TARGET_URL)
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul.imgList.type2 li")))
        items = driver.find_elements(By.CSS_SELECTOR, "ul.imgList.type2 > li")
        print(f"발견된 게시글 개수: {len(items)}개")

        for i in range(len(items)): 
            try:
                current_items = driver.find_elements(By.CSS_SELECTOR, "ul.imgList.type2 > li")
                item = current_items[i]

                try:
                    title_text = item.find_element(By.CSS_SELECTOR, "div span").text
                except:
                    title_text = f"Unknown_Title_{i}"
                
                print(f"[{i+1}/{len(items)}] 처리 중: {title_text}")

                # 이미지 구조상 '바로보기'와 '다운로드' 링크가 따로 있을 수 있으므로 필터링 필요
                download_links = item.find_elements(By.CSS_SELECTOR, "a[href*='download.do']")
                
                if not download_links:
                    print(f"   -> 다운로드 링크를 찾을 수 없습니다.")
                    continue

                for link in download_links:
                    try:
                        file_name = link.get_attribute("title") or title_text
                        print(f"   -> 다운로드 실행: {file_name}")
                        
                        driver.execute_script("arguments[0].click();", link)
                        time.sleep(3) 

                        try:
                            alert = driver.switch_to.alert
                            print(f"   ⚠️ 서버 알림: {alert.text}")
                            alert.accept()
                        except NoAlertPresentException:
                            pass

                    except Exception as e:
                        print(f"   ❌ 개별 파일 다운로드 중 오류: {e}")

            except Exception as e:
                print(f"[{i+1}번 항목] 처리 중 에러 발생: {e}")
                continue

        print(f"\n✅ 작업 완료! 파일 확인: {SAVE_DIR}")

    finally:
        time.sleep(5)
        driver.quit()

if __name__ == "__main__":
    start_download()

#### 파이썬-테서랙트를 이용한 OCR 및 txt 파일로 변환

In [ ]:
params = ['국기연_발간물', '국방부_정책자료', '국방위_국정감사', '국방위_보도', '국방위_회의록']
BASE_SOURCE_DIR = f"C:/Users/user/OneDrive/Desktop/Desktop/VSCode/크롤링프로젝트/"   # PDF들이 흩어져 있는 최상위 폴더: 크롤링프로젝트 -> 필요 시 루트 폴더 수정 필요
BASE_OUTPUT_DIR = "변환됨"
MERGED_FILE = "merged1230.txt"

# Tesseract 실행 경로 설정
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def get_pdf_text(path):
    """텍스트 추출을 먼저 시도하고, 안되면 OCR을 진행하는 함수"""
    full_text = ""
    
    # 1) pdfplumber로 텍스트 직접 추출 시도
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                full_text += t + "\n"
    
    # 2) 만약 추출된 텍스트가 너무 적다면 (스캔된 이미지 PDF일 경우) OCR 실행
    if len(full_text.strip()) < 50:  # 기준은 조정 가능
        print(f"      > 텍스트가 없거나 적음. OCR 진행 중: {os.path.basename(path)}")
        try:
            # PDF를 이미지로 변환 (기본 200dpi)
            images = convert_from_path(path)
            full_text = ""
            for img in images:
                text = pytesseract.image_to_string(img, lang="kor+eng")
                full_text += text + "\n"
        except Exception as e:
            print(f"      ❌ OCR 에러: {e}")
            
    return full_text

with open(MERGED_FILE, "w", encoding="utf-8") as merged_out:
    
    for category in params:
        source_path = os.path.join(BASE_SOURCE_DIR, category)
        output_path = os.path.join(BASE_OUTPUT_DIR, category)
        
        if not os.path.exists(source_path):
            print(f"⚠️ 폴더 없음: {source_path}")
            continue

        os.makedirs(output_path, exist_ok=True)
        print(f"\n📂 '{category}' 카테고리 작업 시작...")

        for root, dirs, files in os.walk(source_path):
            for file in files:
                if file.lower().endswith(".pdf"):
                    src_path = os.path.join(root, file)
                    file_name_only = os.path.splitext(file)[0]
                    dst_txt_path = os.path.join(output_path, f"{file_name_only}.txt")

                    print(f"   📄 처리 중: {file}")

                    content = get_pdf_text(src_path)

                    with open(dst_txt_path, "w", encoding="utf-8") as f:
                        f.write(content)

                    merged_out.write(f"\n\n===== < {category} / {file} > =====\n")
                    merged_out.write(content)

print(f"\n✨ 모든 작업 완료!")
print(f"- 개별 TXT: {BASE_OUTPUT_DIR} 폴더 내부")
print(f"- 통합 TXT: {MERGED_FILE}")

#### '국방부_정책자료' 폴더에 있는 자료들의 경우 pdf 파일 미인식 -> OCR 과정 필요
 - poppler 이용해 추가 변환 처리

In [ ]:
PDF_DIR = "국방부_정책자료"
OUT_DIR = "변환됨"
os.makedirs(OUT_DIR, exist_ok=True)

POPPLER = r"C:\poppler-25.12.0\Library\bin"

# Tesseract 실행 경로 설정
pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'

for file in os.listdir(PDF_DIR):
    if not file.endswith(".pdf"):
        continue

    pdf_path = os.path.join(PDF_DIR, file)
    
    category_dir = os.path.join(OUT_DIR, "국방부_정책자료")
    os.makedirs(category_dir, exist_ok=True)

    out_path = os.path.join(category_dir, file.replace(".pdf", ".txt"))

    print(f"OCR 처리 중: {file}")

    images = convert_from_path(
    pdf_path,
    poppler_path=r"C:\poppler-25.12.0\Library\bin"
    )

    with open(out_path, "w", encoding="utf-8") as f:
        for img in images:
            text = pytesseract.image_to_string(
                img, lang="kor+eng"
            )
            f.write(text + "\n")
